In [1]:
import torch
import torch.nn as nn

In [2]:
import torch.optim as optim

In [3]:
# 로컬에 받은 Fruits-360 경로 지정
# 예: C:/Users/user/Desktop/Myproject/Dataset/Fruit-Images-Dataset
path = r"C:/Users/user/Desktop/Myproject/Dataset/Fruit-Images-Dataset"
print("dataset root:", path)


dataset root: C:/Users/user/Desktop/Myproject/Dataset/Fruit-Images-Dataset


In [4]:
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
import numpy as np


In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(777)

if device == 'cuda':
    torch.cuda.manual_seed_all(777)
    print('cuda')

cuda


In [6]:
# 데이터셋 경로 탐색 및 사용할 클래스 선택
train_candidates = [
    "fruits-360/Training",
    "Training",
    "fruits-360_dataset/Training",
]
test_candidates = [
    "fruits-360/Test",
    "Test",
    "fruits-360_dataset/Test",
]
base = Path(path)
train_root = next((base / c for c in train_candidates if (base / c).exists()), None)
test_root = next((base / c for c in test_candidates if (base / c).exists()), None)

if train_root is None or test_root is None:
    raise FileNotFoundError(f"Training/Test folders not found under {base}")

# 원하는 클래스만 지정 (추가/변경 가능)
allowed_classes = [
    "Apple Crimson Snow",
    "Banana",
    "Blueberry",
    "Orange",
]
print(f"train_root: {train_root}")
print(f"test_root: {test_root}")
print(f"selected classes: {allowed_classes}")


train_root: C:\Users\user\Desktop\Myproject\Dataset\Fruit-Images-Dataset\Training
test_root: C:\Users\user\Desktop\Myproject\Dataset\Fruit-Images-Dataset\Test
selected classes: ['Apple Crimson Snow', 'Banana', 'Blueberry', 'Orange']


In [7]:
# RGB 평균값만 추출하는 커스텀 Dataset
class RGBFruitDataset(Dataset):
    def __init__(self, root_dir, allowed=None):
        self.root = Path(root_dir)
        class_dirs = sorted([d for d in self.root.iterdir() if d.is_dir()])
        if allowed:
            class_dirs = [d for d in class_dirs if d.name in allowed]
        self.classes = [d.name for d in class_dirs]
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.samples = []
        for d in class_dirs:
            label = self.class_to_idx[d.name]
            for img_path in d.glob("*"):
                if img_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                    self.samples.append((img_path, label))
        if not self.samples:
            raise RuntimeError("No images found for the selected classes.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        arr = np.array(img, dtype=np.float32).reshape(-1, 3)
        rgb_mean = arr.mean(axis=0) / 255.0  # 0~1 정규화
        features = torch.tensor(rgb_mean, dtype=torch.float32)
        target = torch.tensor(label, dtype=torch.long)
        return features, target


In [8]:
# DataLoader 구성
batch_size = 256

train_ds = RGBFruitDataset(train_root, allowed_classes)
test_ds = RGBFruitDataset(test_root, allowed_classes)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

num_classes = len(train_ds.classes)
print(f"classes ({num_classes}): {train_ds.classes}")


classes (4): ['Apple Crimson Snow', 'Banana', 'Blueberry', 'Orange']


In [9]:
# MLP 모델 정의 (RGB 3차원 입력)
class RGBFruitMLP(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.net(x)

model = RGBFruitMLP(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)


In [10]:
# 평가 함수

def evaluate(loader):
    model.eval()
    total = 0
    correct = 0
    loss_sum = 0.0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            out = model(x)
            loss = criterion(out, y)
            loss_sum += loss.item() * y.size(0)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return loss_sum / total, correct / total

# 학습 루프
num_epochs = 40
train_losses = []
val_losses = []
val_accs = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * y.size(0)

    train_loss = running_loss / len(train_ds)
    val_loss, val_acc = evaluate(test_loader)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | val_acc {val_acc*100:.2f}%")


Epoch 01 | train_loss 1.3629 | val_loss 1.3214 | val_acc 26.43%
Epoch 02 | train_loss 1.3178 | val_loss 1.2699 | val_acc 26.43%
Epoch 04 | train_loss 1.1933 | val_loss 1.1096 | val_acc 74.04%
Epoch 06 | train_loss 1.0199 | val_loss 0.8967 | val_acc 79.46%
Epoch 08 | train_loss 0.8136 | val_loss 0.6846 | val_acc 93.63%
Epoch 10 | train_loss 0.6354 | val_loss 0.5117 | val_acc 95.86%
Epoch 12 | train_loss 0.4892 | val_loss 0.3912 | val_acc 95.86%
Epoch 14 | train_loss 0.3817 | val_loss 0.3105 | val_acc 99.52%
Epoch 16 | train_loss 0.3085 | val_loss 0.2561 | val_acc 100.00%
Epoch 18 | train_loss 0.2460 | val_loss 0.2101 | val_acc 100.00%
Epoch 20 | train_loss 0.2084 | val_loss 0.1664 | val_acc 100.00%
Epoch 22 | train_loss 0.1721 | val_loss 0.1386 | val_acc 100.00%
Epoch 24 | train_loss 0.1432 | val_loss 0.1078 | val_acc 100.00%
Epoch 26 | train_loss 0.1165 | val_loss 0.0936 | val_acc 100.00%
Epoch 28 | train_loss 0.0994 | val_loss 0.0858 | val_acc 100.00%
Epoch 30 | train_loss 0.0896 | va

In [11]:
# 학습 완료 후 저장
state = {
    "model_state": model.state_dict(),
    "classes": train_ds.classes,
    "class_to_idx": train_ds.class_to_idx,
    "train_losses": train_losses,
    "val_losses": val_losses,
    "val_accs": val_accs,
}
torch.save(state, "rgb_fruit_mlp.pth")
print("model saved -> rgb_fruit_mlp.pth")


model saved -> rgb_fruit_mlp.pth


In [12]:
# 추가 평가: 전체 정확도와 클래스별 정확도
model.eval()
correct = 0
total = 0
per_class_correct = [0 for _ in range(num_classes)]
per_class_total = [0 for _ in range(num_classes)]

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)
        for t, p in zip(y, preds):
            per_class_total[t.item()] += 1
            if t == p:
                per_class_correct[t.item()] += 1

overall_acc = 100.0 * correct / total
print(f"Test Accuracy: {overall_acc:.2f}%")

print("Per-class Accuracy:")
for cls, c_total, c_correct in zip(train_ds.classes, per_class_total, per_class_correct):
    acc = 100.0 * c_correct / c_total if c_total > 0 else 0.0
    print(f"  {cls:20s}: {acc:5.2f}% ({c_correct}/{c_total})")


Test Accuracy: 100.00%
Per-class Accuracy:
  Apple Crimson Snow  : 100.00% (148/148)
  Banana              : 100.00% (166/166)
  Blueberry           : 100.00% (154/154)
  Orange              : 100.00% (160/160)


In [13]:
# 아두이노 시리얼 통신용 패키지 설치
!pip install pyserial


In [14]:
import serial
import time

SERIAL_PORT = 'COM4'  # 아두이노 포트 확인 후 수정
BAUD_RATE = 9600

# 저장된 모델 불러오기
checkpoint = torch.load("rgb_fruit_mlp.pth", map_location=device)
loaded_model = RGBFruitMLP(num_classes=len(checkpoint['classes'])).to(device)
loaded_model.load_state_dict(checkpoint['model_state'])
loaded_model.eval()

classes = checkpoint['classes']
print(f"Model loaded. Classes: {classes}")


Model loaded. Classes: ['Apple Crimson Snow', 'Banana', 'Blueberry', 'Orange']


In [18]:
# 아두이노와 실시간 통신 (무한 루프)
# 아두이노에서 "R,G,B" 형식으로 전송하면 추론 결과 반환
# 예: "120,200,50" → PC가 추론 → "Orange" 출력

try:
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
    time.sleep(2)  # 아두이노 초기화 대기
    print(f"Connected to {SERIAL_PORT} at {BAUD_RATE} baud")
    print("Waiting for RGB data from Arduino...")
    print("Press Ctrl+C to stop.\n")
    
    while True:
        if ser.in_waiting > 0:
            line = ser.readline().decode('utf-8').strip()
            print(f"Received: {line}")
            
            try:
                # RGB 값 파싱 (여러 형식 지원)
                # 형식1: "120,200,50" (CSV)
                # 형식2: "R: 120   G: 200   B: 50" (디버그 형식)
                
                if ',' in line and 'R:' not in line:
                    # CSV 형식
                    r, g, b = map(float, line.split(','))
                else:
                    # 디버그 형식 파싱
                    parts = line.split()
                    r = float(parts[parts.index('R:') + 1]) if 'R:' in parts else 0
                    g = float(parts[parts.index('G:') + 1]) if 'G:' in parts else 0
                    b = float(parts[parts.index('B:') + 1]) if 'B:' in parts else 0
                
                # 0~255 → 0~1 정규화
                rgb_normalized = torch.tensor([r/255.0, g/255.0, b/255.0], dtype=torch.float32).to(device)
                
                # 모델 추론
                with torch.no_grad():
                    logits = loaded_model(rgb_normalized.unsqueeze(0))
                    pred_idx = logits.argmax(1).item()
                    pred_class = classes[pred_idx]
                    confidence = torch.softmax(logits, dim=1)[0][pred_idx].item() * 100
                
                result = f"Prediction: {pred_class} ({confidence:.2f}%)"
                print(result)
                
                # 아두이노로 결과 전송 (선택)
                ser.write(f"{pred_class}\n".encode('utf-8'))
                print("-" * 50)
                
            except ValueError:
                print("Invalid format. Expected 'R,G,B' (e.g., 120,200,50)")
        
        time.sleep(0.1)

except serial.SerialException as e:
    print(f"Serial error: {e}")
    print(f"Please check if Arduino is connected to {SERIAL_PORT}")
except KeyboardInterrupt:
    print("\nStopped by user")
finally:
    if 'ser' in locals() and ser.is_open:
        ser.close()
        print("Serial port closed")


Connected to COM4 at 9600 baud
Waiting for RGB data from Arduino...
Press Ctrl+C to stop.

Received: TCS34725 Ready
Prediction: Apple Crimson Snow (59.61%)
--------------------------------------------------
Received: 106,87,38
Prediction: Apple Crimson Snow (95.22%)
--------------------------------------------------
Received: Result: Apple Crimson Snow
Prediction: Apple Crimson Snow (59.61%)
--------------------------------------------------
Received: 101,71,32
Prediction: Apple Crimson Snow (97.79%)
--------------------------------------------------
Received: Result: Apple Crimson Snow
Prediction: Apple Crimson Snow (59.61%)
--------------------------------------------------
Received: 102,73,33
Prediction: Apple Crimson Snow (97.57%)
--------------------------------------------------
Received: Result: Apple Crimson Snow
Prediction: Apple Crimson Snow (59.61%)
--------------------------------------------------
Received: 46,36,16
Prediction: Apple Crimson Snow (92.66%)
-----------------